# VideoTool — FULL RENDER on Kaggle (config-driven, no per-episode push)

Renders a whole audio-story episode on the T4 GPU (NVENC). Claude Code CLI authors `creative.yaml`
+ a small `render_job.json` and stages both on Drive; **this notebook code never changes per episode**
— it reads the job paths from `gdrive:_VIDEOTOOL_SHARED/render_job.json` at run time. No LLM runs here.

## One-time setup (do ONCE, ever)
1. On your machine: `base64 -w0 ~/.config/rclone/rclone.conf` -> copy the single line.
2. Kaggle: Add-ons -> Secrets -> add `RCLONE_CONF` = that base64 line, toggle **Attached**.
3. Right panel -> Session options -> Accelerator = **GPU T4 x2** (NOT None / P100 — P100 has no NVENC).
4. Save Version -> **Save & Run All (Commit)**.

## Every episode after that (NO `kaggle kernels push`, so the secret stays attached)
- Claude Code CLI writes `render_job.json` to Drive (source/output/checkpoint[/creative]).
- You open THIS saved kernel and click **Save & Run All**. That's it — no secret re-toggle, no path edits.
- Rerun after a disconnect -> resumes from the Drive checkpoint (clips are not re-rendered).


In [ ]:
import os, subprocess, base64
from kaggle_secrets import UserSecretsClient
try:
    raw = UserSecretsClient().get_secret('RCLONE_CONF').strip()
except Exception as e:
    raise RuntimeError(
        'RCLONE_CONF secret is not attached to this kernel. `kaggle kernels push` does NOT inherit the attachment — in the Kaggle UI open Add-ons -> Secrets -> toggle RCLONE_CONF Attached ON, then Save & Run All. (Original error: ' + repr(e) + ')')

try:
    conf = base64.b64decode(raw, validate=True).decode('utf-8'); assert '[gdrive]' in conf
except Exception:
    conf = raw
os.makedirs(os.path.expanduser('~/.config/rclone'), exist_ok=True)
open(os.path.expanduser('~/.config/rclone/rclone.conf'), 'w').write(conf)
subprocess.run('command -v rclone >/dev/null || (curl -s https://rclone.org/install.sh | sudo bash)', shell=True)
remotes = subprocess.run(['rclone', 'listremotes'], capture_output=True, text=True).stdout.strip()
print('rclone remotes:', remotes or '(NONE)', '| conf has [gdrive]:', '[gdrive]' in conf)
assert 'gdrive:' in remotes.split(), 'RCLONE_CONF has no [gdrive] remote (set it to base64 of rclone.conf).'
SHARED = 'gdrive:_VIDEOTOOL_SHARED'
for mod in ('videotool_cloud.py', 'cloud_director.py', 'cloud_render_runner.py'):
    subprocess.run(['rclone', 'copyto', f'{SHARED}/{mod}', mod], check=True)
for lib in ('sfx', 'overlays'):
    dst = os.path.expanduser(f'~/.local/share/videotool/{lib}')
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['rclone', 'copy', f'{SHARED}/{lib}', dst, '--fast-list'], check=False)
import cloud_render_runner as rr
print('setup OK -> ready to render')

In [ ]:
# Reads the job paths from Drive so this cell is identical for every episode (no re-push -> the
# RCLONE_CONF secret is never detached). Claude Code CLI writes render_job.json per episode.
import json, subprocess
CONFIG_REMOTE = 'gdrive:_VIDEOTOOL_SHARED/render_job.json'
raw = subprocess.run(['rclone', 'cat', CONFIG_REMOTE], capture_output=True, text=True).stdout.strip()
if not raw:
    raise RuntimeError(
        f'No render job config at {CONFIG_REMOTE}. The Claude Code CLI writes it per episode '
        '(a small JSON with source/output/checkpoint[/creative]). Nothing to render.')
cfg = json.loads(raw)
for k in ('source', 'output', 'checkpoint'):
    if not cfg.get(k):
        raise RuntimeError(f'render_job.json is missing required key: {k!r}')
print('Rendering:', cfg['source'])
print('     -> output:', cfg['output'])
print('     -> checkpoint:', cfg['checkpoint'])
rr.render_job(
    cfg['source'], cfg['output'], cfg['checkpoint'],
    creative_remote=cfg.get('creative'),
    repo_ref=cfg.get('repo_ref') or 'git+https://github.com/pnd4189/video-tool@main',
    allow_cpu=bool(cfg.get('allow_cpu', False)),
    local_job='/tmp/job',
)
